# Musanga NB08 v4_003 — S2 spectral evidence, annual-vs-seasonal attribution, and **true multiresolution transfer**

This version preserves the original 50 m V3.1 experiment and the v4_002 spectral/temporal analysis, and **adds an executable, restartable spatial-scale experiment** in Sections 08–10.

**Confirmed upstream consistency:** the existing V3.1 10 m UAV-teacher raster was previously reaggregated to the frozen 50 m teacher with MAE <0.002 percentage points and correlation essentially 1 in all four forests (11A teacher consistency report). We validate source paths and that comparison again before proceeding.

**Important:** evaluate **10, 25, 30, 50, 75, 100 m** on matched geographical support, with true pixel-area raster aggregation, S2 model predictors **annual reflectance + vegetation indices**, and strict four-forest LOFO. A 50 m cell cannot be interpolated to generate genuine 10 m labels, and grid sizes at 25/30/75 m are *noninteger multiples* of source 10 m, so full area-weighted reprojection is used. Only 50 m reference support and teacher-valid regions are accepted.

**Run order:** 00–07 (baseline and seasonal diagnostics) then 08–10 (true scale benchmark). For efficiency, change `SCALES=[10,25,30,50,75,100]` in Section 08 to a pilot subset. No new satellite or UAV inference; CPU only. Each scale/site/model caches its artifacts. **LOFO validates geographic transfer, not causal spectral identification of Musanga.**

## 00 — Configuration and exact cached reference audit

In [ ]:
%pip -q install pyarrow scikit-learn scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
ROOTS=[
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
 Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
found=[p for p in ROOTS if (p/'frozen_50m_reference.parquet').is_file()]
assert len(found)==1,('Expected one existing frozen reference; no new downloads',found)
ROOT=found[0];OUT=ROOT/'17_NB08_V3_1_detectability'
OUT.mkdir(parents=True,exist_ok=True)
D=pd.read_parquet(ROOT/'frozen_50m_reference.parquet')
SITES=('Baego','MOS','Yangambi','Yoko')
KEYS=['site','mosaic','row','col']
assert len(D)==14454 and not D.duplicated(KEYS).any()
assert D.groupby('site').size().to_dict()=={'Baego':480,'MOS':11494,'Yangambi':1383,'Yoko':1097}
assert D.target_fraction.between(0,1).all()
assert {'x','y','target_fraction','site'}.issubset(D.columns)
ANNUAL=sorted(c for c in D if c.startswith('RAW_ANNUAL__'))
QUARTER=sorted(c for c in D if c.startswith('S2_QUARTERLY_3YR__'))
EMB=sorted(c for c in D if c.startswith('EMB_ANNUAL__'))
assert len(ANNUAL)>0 and len(QUARTER)==44 and len(EMB)==64,(len(ANNUAL),len(QUARTER),len(EMB))
# Quarterly clear-observation counts are not spectral reflectance.
Q_COUNTS=[f'S2_QUARTERLY_3YR__{b:02d}' for b in (11,22,33,44)]
assert set(Q_COUNTS).issubset(QUARTER)
S2=[*ANNUAL,*[c for c in QUARTER if c not in Q_COUNTS]]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':[ *S2,*EMB ]}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
assert np.isfinite(D[ALL_FEATURES+['target_fraction','x','y']].to_numpy('float32')).all()
assert (D[Q_COUNTS].to_numpy(float)>=1).all(), 'Frozen reference has quarterly no-observation cells; inspect source.'
SEED=20261120
CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
           min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
# Compute fingerprints once; do not overwrite unrelated historical outputs.
SOURCE_SHA=hashlib.sha256(pd.util.hash_pandas_object(
 D[KEYS+['x','y','target_fraction']+ALL_FEATURES],index=False).values.tobytes()).hexdigest()
CONFIG={'version':'NB08_V3_1_LOFO_detectability_v4_001','sites':SITES,
 'n':len(D),'features':{k:len(v) for k,v in FEATURE_SETS.items()},
 'seed':SEED,'cap':CAP_PER_TRAIN_FOREST,'model':MODEL,
 'thresholds':THRESHOLDS,'source_sha256':SOURCE_SHA}
SIGNATURE=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:14]
RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
config_path=RUN/'config.json'
if config_path.is_file():assert json.loads(config_path.read_text())==json.loads(json.dumps(CONFIG))
else:config_path.write_text(json.dumps(CONFIG,indent=2))
print('Source:',ROOT/'frozen_50m_reference.parquet')
print('Saved outputs:',RUN,'| signature:',SIGNATURE)
display(D.groupby('site').target_fraction.agg(n='size',mean='mean',std='std',maximum='max').round(3))
print('Predictors: S2 annual',len(ANNUAL),'+ quarterly reflectance',len(QUARTER)-len(Q_COUNTS),'; embeddings',len(EMB))
print('No imagery downloads or raster reads are required.')


## 01 — A strict, small, cached leave-one-forest-out experiment

Use the original V3.1 source's **balanced-by-forest and cover-stratum** training principle; training selections depend only on training-forest targets. Each held-out forest is untouched until scoring. Single-model fits are intended as a first detectability test rather than hyperparameter optimization.

In [ ]:
y=D.target_fraction.to_numpy(np.float32)
site=D.site.to_numpy()
def balanced_draw(pool,heldout):
 rng=np.random.default_rng(SEED+sum(map(ord,heldout)))
 chosen=[]
 for forest in SITES:
  ix=pool[site[pool]==forest]
  if not len(ix):continue
  cap=min(len(ix),CAP_PER_TRAIN_FOREST)
  if len(ix)<=cap:
   chosen.extend(ix.tolist());continue
  strata=np.digitize(y[ix],[.10,.20,.40])
  bins={j:ix[strata==j] for j in range(4)}
  pick=[]
  # Keep up to a quarter of training sample above 40% where available.
  take=min(len(bins[3]),int(np.ceil(.25*cap)))
  if take:
   ids=rng.choice(bins[3],take,replace=False)
   pick.extend(ids.tolist());bins[3]=np.setdiff1d(bins[3],ids)
  while len(pick)<cap:
   active=[k for k in bins if len(bins[k])]
   assert active
   for k in active:
    n=min(len(bins[k]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
    if n>0:
     ids=rng.choice(bins[k],n,replace=False)
     pick.extend(ids.tolist());bins[k]=np.setdiff1d(bins[k],ids)
    if len(pick)>=cap:break
  chosen.extend(pick)
 return np.asarray(sorted(chosen),dtype=int)
predictions=[]
audit=[]
for held in SITES:
 test=np.flatnonzero(site==held)
 train_pool=np.flatnonzero(site!=held)
 train=balanced_draw(train_pool,held)
 assert not np.intersect1d(train,test).size
 audit.append(dict(heldout=held,train_cells=len(train),test_cells=len(test),
                   high_train=int((y[train]>=.40).sum())))
 # The constant null is learned from the **same** training draw.
 null_value=float(np.mean(y[train]))
 frame=D.iloc[test][KEYS+['x','y','target_fraction']].copy().reset_index(drop=True)
 frame['heldout']=held
 frame['M0_mean']=null_value
 for model_name,cols in FEATURE_SETS.items():
  cache=RUN/f'{held}_{model_name}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test_index'],test)
    assert z['source_sha256'].item()==SOURCE_SHA
    pred=z['pred'].astype(np.float32)
   print('CACHE',held,model_name,flush=True)
  else:
   t=time.monotonic()
   model=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   model.fit(D.iloc[train][cols].to_numpy('float32'),y[train])
   pred=np.clip(model.predict(D.iloc[test][cols].to_numpy('float32')),0,1).astype(np.float32)
   tmp=cache.with_suffix('.partial')
   with open(tmp,'wb') as f:
    np.savez_compressed(f,test_index=test,pred=pred,source_sha256=SOURCE_SHA)
   os.replace(tmp,cache)
   print('FIT',held,model_name,'seconds',round(time.monotonic()-t,1),flush=True)
  frame[model_name]=pred
 predictions.append(frame)
P=pd.concat(predictions,ignore_index=True)
assert len(P)==len(D) and not P.duplicated(KEYS).any()
assert P[['M0_mean',*FEATURE_SETS]].notna().all().all()
P.to_parquet(RUN/'01_paired_LOFO_predictions.parquet',index=False)
pd.DataFrame(audit).to_csv(RUN/'01_training_fold_sizes.csv',index=False)
display(pd.DataFrame(audit))
print('All held-out predictions cached on the same 14,454 frozen reference cells.')


## 02 — Forest-equal continuous-cover prediction and abundance detection

Use per-forest equal-weight metrics, never global pixel weighting that allows MOS to dominate. Continuous regression: R², MAE, RMSE, bias and Spearman. For thresholds 5%, 20%, 40%, compute ROC-AUC and average precision from **continuous prediction scores**, plus prevalence and a training-derived constant-score baseline. Undefined AUC for missing positive or negative classes is reported as NaN (not zero).

In [ ]:
MODELS=['M0_mean',*FEATURE_SETS]
def metrics(g,col):
 obs=g.target_fraction.to_numpy(float)
 pred=g[col].to_numpy(float)
 err=pred-obs
 var=float(np.mean((obs-np.mean(obs))**2))
 rho=spearmanr(obs,pred).statistic if np.std(pred)>1e-12 and np.std(obs)>1e-12 else np.nan
 return dict(n=len(g),R2=float(1-np.mean(err**2)/var) if var>0 else np.nan,
             MAE_pp=float(100*np.mean(np.abs(err))),
             RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
             bias_pp=float(100*np.mean(err)),
             spearman_rho=float(rho) if np.isfinite(rho) else np.nan)
PER=pd.DataFrame([dict(site=forest,model=model,**metrics(g,model))
 for forest,g in P.groupby('site') for model in MODELS])
PER.to_csv(RUN/'02_continuous_by_forest.csv',index=False)
SUMMARY=[]
for model in MODELS:
 sub=PER[PER.model.eq(model)]
 # Global equal-forest R² with pooled equal-forest target variance.
 means=P.groupby('site').target_fraction.mean()
 v=np.mean([np.mean((g.target_fraction-means[s])**2)+(means[s]-means.mean())**2
      for s,g in P.groupby('site')])
 mse=np.mean([np.mean((g[model]-g.target_fraction)**2) for s,g in P.groupby('site')])
 SUMMARY.append(dict(model=model,n_cells=len(P),n_forests=len(SITES),
  equal_forest_R2=float(1-mse/v) if v>0 else np.nan,
  equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
  equal_forest_RMSE_pp=float(np.sqrt(np.mean(sub.RMSE_pp**2))),
  equal_forest_bias_pp=float(sub.bias_pp.mean()),
  mean_forest_rho=float(sub.spearman_rho.mean())))
SUMMARY=pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN/'02_equal_forest_summary.csv',index=False)
DISC=[]
for (forest,model),g in (( (f,m),q) for f,q in P.groupby('site') for m in MODELS):
 obs=g.target_fraction.to_numpy(float)
 pred=g[model].to_numpy(float)
 for t in THRESHOLDS:
  true=(obs>=t).astype(int)
  npos=int(true.sum());nneg=len(true)-npos
  auc=roc_auc_score(true,pred) if npos and nneg and np.unique(pred).size>1 else np.nan
  ap=average_precision_score(true,pred) if npos and nneg else np.nan
  DISC.append(dict(site=forest,model=model,threshold=t,
       n=len(g),positive_n=npos,negative_n=nneg,
       prevalence=npos/len(g),ROC_AUC=auc,average_precision=ap))
DISC=pd.DataFrame(DISC)
DISC.to_csv(RUN/'02_abundance_detection_by_forest.csv',index=False)
DISPLAY_DISC=(DISC.groupby(['model','threshold']).agg(
 n_forests_evaluable=('ROC_AUC','count'),
 mean_AUC=('ROC_AUC','mean'),
 mean_AP=('average_precision','mean'),
 mean_prevalence=('prevalence','mean')).reset_index())
DISPLAY_DISC.to_csv(RUN/'02_equal_forest_detection_summary.csv',index=False)
print('Equal-forest continuous-cover scores:')
display(SUMMARY.round(3))
print('Abundance discrimination (undefined sites omitted, counts explicit):')
display(DISPLAY_DISC.round(3))
print('Important: high-cover detection is undefined for forests without high-cover examples.')


## 03 — Visual evidence: spectral contrasts, held-out predictions and detectability

**Figure A:** within-forest low vs high cover spectral contrast on the frozen 50 m cells; avoids visually conflating broad differences among forests. Spectral columns are raw annual source bands (band numbers until independently verified); do not claim wavelengths without an audited metadata lookup.

**Figure B:** held-out continuous fractional cover by forest, preserving the four-fold transfer test.

**Figure C:** abundance discrimination curves at 5, 20 and 40% (mean of evaluable forests, number of evaluable forests displayed).

In [ ]:
import matplotlib
from matplotlib.lines import Line2D
# A: Within-forest annual S2 contrast. Use percentiles for robustness.
low=.05;high=.40
rows=[]
for forest,g in D.groupby('site'):
 lo=g[g.target_fraction<=low]
 hi=g[g.target_fraction>=high]
 if len(lo)<10 or len(hi)<10:
  print('Spectral contrast not estimable',forest,'low',len(lo),'high',len(hi));continue
 for i,c in enumerate(ANNUAL,1):
  lv=float(lo[c].median());hv=float(hi[c].median())
  rows.append(dict(site=forest,band=c,band_order=i,n_low=len(lo),
                   n_high=len(hi),median_low=lv,median_high=hv,
                   difference=hv-lv))
SPECTRA=pd.DataFrame(rows)
SPECTRA.to_csv(RUN/'03_within_forest_annual_S2_spectra.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=SPECTRA[SPECTRA.site.eq(forest)]
 if len(g):
  ax.plot(g.band_order,g.median_low,'o-',label='≤5% UAV Musanga')
  ax.plot(g.band_order,g.median_high,'o-',label='≥40% UAV Musanga')
  ax.set_title(f'{forest} | low n={g.n_low.iloc[0]}, high n={g.n_high.iloc[0]}')
 else:
  ax.set_title(f'{forest} | too few high/low cells')
 ax.set(xlabel='Annual S2 source band number',ylabel='Median archived band value')
 ax.grid(alpha=.15)
axs.flat[0].legend(fontsize=8)
fig.suptitle('A · Within-forest spectral associations (not species-specific identification)')
fig.savefig(RUN/'03A_S2_within_forest_low_high_spectra.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)
# B: For each forest show S2 vs embeddings+S2; cap plots only, not metrics.
fig,axs=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=P[P.site.eq(forest)].copy()
 if len(g)>1300:g=g.sample(n=1300,random_state=SEED)
 ax.scatter(g.target_fraction*100,g.M1_S2*100,s=9,alpha=.3,label='S2')
 ax.scatter(g.target_fraction*100,g.M3_S2_EMB*100,s=9,alpha=.3,label='S2 + embeddings')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlabel='UAV Musanga cover (%)',
        ylabel='Held-out satellite prediction (%)',
        xlim=(-2,102),ylim=(-2,102))
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('B · Strict leave-one-forest-out prediction (matched reference cells)')
fig.savefig(RUN/'03B_heldout_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# C: Mean AUC by cover threshold and number of forests actually evaluable.
fig,ax=plt.subplots(figsize=(8,5))
for model in MODELS[1:]:
 q=DISPLAY_DISC[DISPLAY_DISC.model.eq(model)].sort_values('threshold')
 ax.plot(q.threshold*100,q.mean_AUC,marker='o',label=model)
 for v in q.itertuples():
  if np.isfinite(v.mean_AUC):
   ax.annotate(f'n={v.n_forests_evaluable}',(v.threshold*100,v.mean_AUC),
               xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
ax.axhline(.5,ls='--',color='gray',lw=1,label='Chance AUC')
ax.set(xlabel='UAV Musanga cover threshold (%)',ylabel='Forest-mean held-out ROC-AUC',
       ylim=(0,1.05),xticks=[5,20,40],
       title='C · Ability to distinguish Musanga-abundant patches')
ax.legend(fontsize=9);ax.grid(alpha=.15)
fig.savefig(RUN/'03C_abundance_discrimination.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Three figures saved:',RUN)


## 04 — Compact decision and caveats

Interpret performance for **all four forests**, not just the pooled R². Compare M1 vs M0 (satellite information); M2 vs M1 (embeddings versus raw S2); and M3 vs both. AUC > 0.5 on evaluable, held-out forests supports **association**, not Musanga-specific spectral identifiability. Inspect forest abundance prevalence: at >40% the Yangambi fold has no positives, so the ROC-AUC is not defined.

**Caveats:** frozen V3.1 labels derive from an earlier UAV model; pixel-neighbour dependence invalidates naive cell-wise confidence intervals. Sampling to balance training forests is fixed, not evaluated as an optimal algorithm. The threshold experiment uses continuous regression scores—not binary models—avoiding separate threshold-fitting. Do not interpret within-forest band profiles as independent species spectra; examine confounding by young forest and disturbance in a later experiment.

**Next iteration:** once NB07 V4 RGB+MS inference is complete, reaggregate its valid-pixel predictions to the exact 50 m satellite cells, keep a matched validity mask, and repeat the same fixed experiment. Do not mix V3.1 and V4 targets in a single score.

In [ ]:
manifest={'experiment':'NB08_v4_001_V3_1','input':str(ROOT/'frozen_50m_reference.parquet'),
 'source_sha256':SOURCE_SHA,'signature':SIGNATURE,
 'n_reference_cells':len(D),'n_sites':len(SITES),
 'feature_families':{k:len(v) for k,v in FEATURE_SETS.items()},
 'spatial_validation':'leave_one_forest_out',
 'reference':'V3_1 UAV teacher-derived fractional Musanga cover',
 'warning':'Predictive association is not species-unique identification',
 'outputs':str(RUN)}
(RUN/'04_run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('NB08 V3.1 detection experiment completed:',RUN)
print('Primary output: 02_equal_forest_summary.csv')
print('Detection output: 02_equal_forest_detection_summary.csv')
print('Paired predictions: 01_paired_LOFO_predictions.parquet')


## 05 — Proper Sentinel-2 spectra and seasonal separability

Plot only reflectance bands at their physical wavelength locations, not NDVI/NDMI/counts. Within each forest, compare ≤5% versus ≥40% cover, showing IQR and effect sizes. Quarterly columns are arranged 10 reflectance channels + 1 count for each pooled 3-year quarter. These are not monthly observations.

In [ ]:
# 10 original annual reflectance bands (verified upstream Earth Engine selection order)
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
S2_WAVELENGTH_NM=np.array([492,560,665,833,704,740,783,865,1610,2190])
S2_ORDER=np.argsort(S2_WAVELENGTH_NM)
assert len(ANNUAL)==15 and len(QUARTER)==44
REFLECTANCE=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
INDICES=[f'RAW_ANNUAL__{i:02d}' for i in range(11,15)]
CLEAR_ANNUAL='RAW_ANNUAL__15'
Q_REFLECTANCE={q:[f'S2_QUARTERLY_3YR__{(q-1)*11+i:02d}' for i in range(1,11)] for q in range(1,5)}
assert set(REFLECTANCE+INDICES+[CLEAR_ANNUAL])==set(ANNUAL)
assert set(sum(Q_REFLECTANCE.values(),[])+Q_COUNTS)==set(QUARTER)
# Plot median and interquartile envelope of raw reflectance; no modelling in this block.
spectral_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(15,10),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast\\nlow={len(low)}, high={len(high)}',
          ha='center',va='center',transform=ax.transAxes)
  ax.set_title(f'{forest} — not estimable')
  continue
 for subset,label,color in [(low,'≤5% Musanga','steelblue'),(high,'≥40% Musanga','darkorange')]:
  arr=subset[REFLECTANCE].to_numpy(float)[:,S2_ORDER]
  wl=S2_WAVELENGTH_NM[S2_ORDER]
  med=np.median(arr,axis=0);q25=np.percentile(arr,25,axis=0);q75=np.percentile(arr,75,axis=0)
  ax.plot(wl,med,'o-',label=f'{label}, n={len(subset)}',color=color)
  ax.fill_between(wl,q25,q75,alpha=.16,color=color)
  for j,k in enumerate(S2_ORDER):
   spectral_rows.append(dict(site=forest,group=label,band=S2_BANDS[k],
     wavelength_nm=int(S2_WAVELENGTH_NM[k]),n=len(subset),
     median=float(med[j]),q25=float(q25[j]),q75=float(q75[j])))
 ax.set(xlabel='Approximate central wavelength (nm)',
        ylabel='S2 annual median surface reflectance',title=forest)
 ax.legend(fontsize=8);ax.grid(alpha=.12)
fig.suptitle('True annual S2 reflectance: low vs high 50 m Musanga cover (median ± IQR)')
fig.savefig(RUN/'05A_annual_S2_true_reflectance_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(spectral_rows).to_csv(RUN/'05A_annual_S2_true_reflectance_IQR.csv',index=False)
# Seasonal reflectance: 3-year pooled quarter medians, NOT single-year snapshots.
seasonal_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast: high={len(high)}',ha='center',va='center',transform=ax.transAxes)
  ax.set_title(forest);continue
 for q in range(1,5):
  for subset,name in [(low,'low'),(high,'high')]:
   # NDVI is computed from quarter median reflectances (B8 = index 4; B4 = index 3).
   nir=subset[Q_REFLECTANCE[q][3]].to_numpy(float)
   red=subset[Q_REFLECTANCE[q][2]].to_numpy(float)
   ndvi=(nir-red)/np.maximum(nir+red,1e-8)
   med=float(np.median(ndvi))
   seasonal_rows.append(dict(site=forest,quarter=q,class_name=name,
                              n=len(subset),median_NDVI=med,
                              q25_NDVI=float(np.percentile(ndvi,25)),
                              q75_NDVI=float(np.percentile(ndvi,75))))
 for name,color in [('low','steelblue'),('high','darkorange')]:
  a=pd.DataFrame(seasonal_rows)
  a=a[(a.site==forest)&(a.class_name==name)].sort_values('quarter')
  ax.plot(a.quarter,a.median_NDVI,'o-',color=color,label=name)
  ax.fill_between(a.quarter.to_numpy(float),a.q25_NDVI.to_numpy(float),
                  a.q75_NDVI.to_numpy(float),alpha=.15,color=color)
 ax.set(title=forest,xlabel='Pooled 3-year calendar quarter',ylabel='Quarterly median-reflectance NDVI',
        xticks=[1,2,3,4])
 ax.grid(alpha=.15);ax.legend()
fig.suptitle('Seasonal spectral association: quarter medians pooled across three years')
fig.savefig(RUN/'05B_pooled_quarterly_NDVI_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(seasonal_rows).to_csv(RUN/'05B_pooled_quarterly_NDVI_IQR.csv',index=False)
print('Spectral plots complete. Overlap of distributions is expected in mixed 50 m canopies.')


## 06 — Why does Sentinel-2 predict so well? LOFO annual-versus-seasonal ablation

This isolates single-year reflectance, four annual indices, and 3-year pooled quarterly reflectance. Identical targets, same training draws (from Section 01), same holdouts and model hyperparameters. It does **not** infer phenological mechanisms from predictive performance alone. All metrics are computed with equal weight for each forest.

In [ ]:
ABLATION={
 'A_reflectance_only':REFLECTANCE,
 'B_annual_plus_indices':REFLECTANCE+INDICES,
 'C_pooled_quarters_only':sum(Q_REFLECTANCE.values(),[]),
 'D_annual_plus_quarters':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[]),
 'E_embeddings_only':EMB,
 'F_all_plus_embeddings':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[])+EMB
}
ABL_CFG={'version':'NB08_S2_temporal_ablation_v1','seed':SEED,'model':MODEL,
        'train_cap':CAP_PER_TRAIN_FOREST,'features':ABLATION,
        'source':SOURCE_SHA}
ABL_ID=hashlib.sha256(json.dumps(ABL_CFG,sort_keys=True).encode()).hexdigest()[:12]
ABL_DIR=OUT/'18_S2_temporal_ablation'/ABL_ID
ABL_DIR.mkdir(parents=True,exist_ok=True)
(ABL_DIR/'config.json').write_text(json.dumps(ABL_CFG,indent=2))
rows=[];pred_frames=[]
for forest in SITES:
 test=np.flatnonzero(site==forest)
 train=balanced_draw(np.flatnonzero(site!=forest),forest)
 part=D.iloc[test][KEYS+['target_fraction']].copy().reset_index(drop=True)
 for variant,features in ABLATION.items():
  cache=ABL_DIR/f'{forest}_{variant}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test'],test)
    predict=z['pred'].astype(float)
   print('CACHE',forest,variant,flush=True)
  else:
   start=time.monotonic()
   fit=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   fit.fit(D.iloc[train][features].to_numpy(np.float32),y[train])
   predict=np.clip(fit.predict(D.iloc[test][features].to_numpy(np.float32)),0,1)
   with open(cache.with_suffix('.partial'),'wb') as f:
    np.savez_compressed(f,test=test,pred=predict.astype(np.float32))
   os.replace(cache.with_suffix('.partial'),cache)
   print('FIT',forest,variant,round(time.monotonic()-start,1),'seconds',flush=True)
  part[variant]=predict
  err=predict-y[test]
  rows.append(dict(site=forest,variant=variant,n=len(test),n_features=len(features),
     MAE_pp=float(100*np.mean(abs(err))),MSE=float(np.mean(err**2)),
     bias_pp=float(100*np.mean(err)),
     R2=float(1-np.mean(err**2)/np.var(y[test])) if np.var(y[test])>0 else np.nan))
 pred_frames.append(part)
ABL_P=pd.concat(pred_frames,ignore_index=True)
ABL_P.to_parquet(ABL_DIR/'06_all_paired_LOFO_predictions.parquet',index=False)
ABL_PER=pd.DataFrame(rows)
ABL_PER.to_csv(ABL_DIR/'06_per_forest.csv',index=False)
forest_mean=D.groupby('site').target_fraction.mean().mean()
target_variance=float(np.mean([
 np.mean((q.target_fraction.to_numpy(float)-forest_mean)**2)
 for _,q in D.groupby('site')]))
summary=ABL_PER.groupby('variant').agg(n_forests=('site','nunique'),
  mean_MAE_pp=('MAE_pp','mean'),mean_MSE=('MSE','mean'),
  mean_bias_pp=('bias_pp','mean'),mean_per_forest_R2=('R2','mean'))
summary['equal_forest_R2']=1-summary.mean_MSE/target_variance
summary['equal_forest_RMSE_pp']=100*np.sqrt(summary.mean_MSE)
summary=summary.reset_index()
summary.to_csv(ABL_DIR/'06_equal_forest_ablation_summary.csv',index=False)
display(summary[['variant','n_forests','mean_MAE_pp','equal_forest_R2',
                 'equal_forest_RMSE_pp','mean_bias_pp']].round(3))
display(ABL_PER.pivot(index='site',columns='variant',values='MAE_pp').round(2))
fig,ax=plt.subplots(figsize=(11,5),layout='constrained')
ax.bar(summary.variant,summary.equal_forest_R2)
ax.axhline(0,color='gray',lw=.8)
ax.tick_params(axis='x',rotation=28)
ax.set(ylabel='Equal-forest R²',title='Does seasonal S2 add transfer beyond annual reflectance and indices?')
fig.savefig(ABL_DIR/'06_S2_annual_vs_seasonal_LOFO.png',dpi=180)
plt.show();plt.close(fig)
print('Annual-versus-seasonal LOFO results:',ABL_DIR)


## 07 — Native multiresolution source audit (corrected folder)

The frozen 50 m table itself cannot be interpolated for a 10–100 m experiment. The V3.1 **10 m UAV-teacher** rasters exist under `CLEAN_NB06/11A_dense_UAV_teacher_10m`. This corrects the wrong parent-directory path in v4_002. The 10 m rasters were previously shown to match the frozen 50 m teacher extremely closely. Sections 08–10 now use these data to do actual area-weighted aggregation and LOFO fitting.

In [ ]:
# Read-only source discovery. Never silently infer multiscale targets from 50m cells.
from glob import glob
BASE=ROOT.parent
teacher_dir=ROOT/'11A_dense_UAV_teacher_10m'
predictor_dir=BASE.parent/'04_satellite_predictors'/'input_layers'
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
scale_rows=[]
for forest in SITES:
 teacher=teacher_dir/f'{forest}_Musanga_UAVteacher_mean_10m.tif'
 annual=predictor_dir/'RAW_ANNUAL'/f'{forest}_RAW_ANNUAL_{YEARS[forest]}_10m.tif'
 embeddings=predictor_dir/'EMB_ANNUAL'/f'{forest}_EMB_ANNUAL_{YEARS[forest]}_10m.tif'
 scale_rows.append(dict(site=forest,teacher_10m_present=teacher.is_file(),
  annual_S2_10m_present=annual.is_file(),embedding_10m_present=embeddings.is_file(),
  teacher_file=str(teacher),S2_file=str(annual),embedding_file=str(embeddings)))
SCALE_INPUT_AUDIT=pd.DataFrame(scale_rows)
display(SCALE_INPUT_AUDIT[['site','teacher_10m_present','annual_S2_10m_present',
                            'embedding_10m_present']])
SCALE_INPUT_AUDIT.to_csv(RUN/'07_multiscale_source_audit.csv',index=False)
if not SCALE_INPUT_AUDIT[['teacher_10m_present','annual_S2_10m_present','embedding_10m_present']].all().all():
 print('Some sources are missing from expected paths. Locate/verify actual cache paths before multi-scale extraction.')
print('PROPOSED genuine analysis resolutions (m):', [10,20,25,30,50,75,100])
print('Scale benchmark NOT run. Need verified harmonized labels, support and identical spatial footprints.')


## 08 — Rebuild genuine 10–100 m cells from the verified V3.1 native inputs

For each scale and forest, reproject **native 10 m teacher probabilities** and **native 10 m Sentinel-2 annual reflectance/indices** using `Resampling.average` onto a common metric raster grid. Reproject source teacher valid/invalid mask separately to retain only ≥90% observed teacher coverage. Restrict evaluation to the actual geographic footprint of the existing, quality-approved 50 m reference cells (center of new cell inside a valid frozen 50 m cell). This yields comparable geographical scope but *not identical cell counts or a fixed sample statistic across scales*.

At 50 m, compare reconstructed teacher directly to the original frozen `target_fraction`; stop on discrepancies >0.5 percentage points MAE (a much looser guard than earlier observed <0.002 pp). At each scale cache the eligible tabular dataset. Never use cubic or bilinear interpolation to synthesize fine-scale ground truth.

**Computational guard:** a bounded rectangle around approved 50 m reference cells is processed rather than huge mosaic extents. Limit per-forest evaluation cells in Section 09, *without changing saved source-derived targets*.

In [ ]:
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.windows import Window
from contextlib import ExitStack
SCALES=[10,25,30,50,75,100] # Genuine target/predictor support, metres
MIN_TEACHER_COVERAGE=.90
MIN_SOURCE_S2_COVERAGE=.90
MAX_EXTRACT_GRID_PIXELS=9_000_000
SCALE_VERSION='NB08_native_teacher_annual_S2_v2'
SCALE_OUT=OUT/'19_native_scale_LOFO'
SCALE_OUT.mkdir(parents=True,exist_ok=True)
SCALE_PATHS={}
for forest in SITES:
 teacher=ROOT/'11A_dense_UAV_teacher_10m'/f'{forest}_Musanga_UAVteacher_mean_10m.tif'
 annual=ROOT.parent.parent/'04_satellite_predictors/input_layers/RAW_ANNUAL'/f'{forest}_RAW_ANNUAL_{{"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}[forest]}_10m.tif'
 # The native S2 sources were already found in Section 07; explicit assertion prevents silent re-extraction.
 assert teacher.is_file(),('Missing native teacher',teacher)
 assert annual.is_file(),('Missing annual S2',annual)
 SCALE_PATHS[forest]={'teacher':teacher,'annual':annual}
manifest=dict(version=SCALE_VERSION,scales=SCALES,teacher_coverage=MIN_TEACHER_COVERAGE,
   s2_coverage=MIN_SOURCE_S2_COVERAGE,source_sha=SOURCE_SHA,
   inputs={forest:{k:(str(p),p.stat().st_size,p.stat().st_mtime_ns)
        for k,p in paths.items()} for forest,paths in SCALE_PATHS.items()})
SCALE_HASH=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:14]
SCALE_RUN=SCALE_OUT/SCALE_HASH
SCALE_RUN.mkdir(parents=True,exist_ok=True)
(SCALE_RUN/'08_source_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Native-scale output:',SCALE_RUN)

def mean_warp(src,dst_transform,h,w,index=1,source_array=None,source_nodata=None):
 target=np.full((h,w),np.nan,dtype=np.float32)
 if source_array is None:
  source=rasterio.band(src,index)
  source_transform=src.transform
  source_crs=src.crs
  if source_nodata is None:source_nodata=src.nodata
 else:
  source=source_array
  source_transform=src.transform
  source_crs=src.crs
 reproject(source,target,src_transform=source_transform,src_crs=source_crs,
   src_nodata=source_nodata,dst_transform=dst_transform,dst_crs=src.crs,
   dst_nodata=np.nan,resampling=Resampling.average,num_threads=2)
 return target

def valid_warp(src,dst_transform,h,w,index=1):
 # Average of binary valid-data indicators: fraction of real source support.
 data=src.read(index,masked=True)
 good=(~np.ma.getmaskarray(data)) & np.isfinite(np.asarray(data.filled(np.nan)))
 arr=good.astype('float32')
 out=np.zeros((h,w),np.float32)
 reproject(arr,out,src_transform=src.transform,src_crs=src.crs,
   src_nodata=None,dst_transform=dst_transform,dst_crs=src.crs,
   dst_nodata=0,resampling=Resampling.average,num_threads=2)
 return out

def build_scale_site(scale,forest):
 out=SCALE_RUN/f'{forest}_{scale}m.parquet'
 if out.is_file():
  cached=pd.read_parquet(out)
  assert cached.scale_m.eq(scale).all() and cached.site.eq(forest).all()
  print('CACHE',forest,scale,'n',len(cached),flush=True)
  return cached
 ref=D[D.site.eq(forest)]
 rr=ref.row.to_numpy(int);cc=ref.col.to_numpy(int)
 with ExitStack() as stack:
  s2=stack.enter_context(rasterio.open(SCALE_PATHS[forest]['annual']))
  teacher=stack.enter_context(rasterio.open(SCALE_PATHS[forest]['teacher']))
  assert s2.crs==teacher.crs and s2.crs.is_projected,(forest,'CRS mismatch')
  assert s2.count>=14,(forest,'annual S2 lacks reflectances/indices')
  assert abs(s2.res[0]-10)<.1 and abs(s2.res[1]-10)<.1,(forest,s2.res)
  assert abs(teacher.res[0]-10)<.1,(forest,'teacher not 10m')
  # Each 50m frozen cell is addressed on the site's 10m Sentinel grid.
  factor=scale/10.
  r0=int(np.floor(rr.min()*50/scale));c0=int(np.floor(cc.min()*50/scale))
  r1=int(np.ceil((rr.max()+1)*50/scale));c1=int(np.ceil((cc.max()+1)*50/scale))
  h=r1-r0;w=c1-c0
  assert h*w<=MAX_EXTRACT_GRID_PIXELS,(forest,scale,'too large grid',h,w)
  dst_transform=s2.transform*Affine.translation(c0*factor,r0*factor)*Affine.scale(factor,factor)
  target=mean_warp(teacher,dst_transform,h,w)
  teach_support=valid_warp(teacher,dst_transform,h,w)
  good=np.isfinite(target)&(target>=0)&(target<=1)&(teach_support>=MIN_TEACHER_COVERAGE)
  # Use only the same approved 50m geographic cell footprints from the frozen reference.
  ref_width=int(cc.max()+1)
  keyset=np.unique(rr.astype(np.int64)*ref_width+cc)
  ygrid,xgrid=np.nonzero(good)
  base_row=np.floor((r0+ygrid+.5)*scale/50).astype(np.int64)
  base_col=np.floor((c0+xgrid+.5)*scale/50).astype(np.int64)
  eligible=np.isin(base_row*ref_width+base_col,keyset,assume_unique=False)
  ygrid=ygrid[eligible];xgrid=xgrid[eligible]
  base_row=base_row[eligible];base_col=base_col[eligible]
  assert len(ygrid)>0,(forest,scale,'No matched native teacher/50m footprint cells')
  # 14 annual reflectance and vegetation indices, computed by true area aggregation.
  feature_data={}
  for j in range(1,15):
   image=mean_warp(s2,dst_transform,h,w,index=j)
   # Select only actual model cells; exclude nodata and outside-source areas.
   feature_data[f'S2_{j:02d}']=image[ygrid,xgrid].copy()
   del image
  df=pd.DataFrame(dict(site=forest,scale_m=scale,row=r0+ygrid,col=c0+xgrid,
     frozen_50m_row=base_row,frozen_50m_col=base_col,
     x=np.asarray((dst_transform*(xgrid+.5,ygrid+.5))[0],float),
     y=np.asarray((dst_transform*(xgrid+.5,ygrid+.5))[1],float),
     target_fraction=target[ygrid,xgrid],
     teacher_coverage=teach_support[ygrid,xgrid],**feature_data))
  df=df[np.isfinite(df[[f'S2_{j:02d}' for j in range(1,15)]].to_numpy(float)).all(axis=1)].copy()
  # Physical reflectances must not be nodata or invalid (indices can be negative).
  df=df[(df[[f'S2_{j:02d}' for j in range(1,11)]]>=0).all(axis=1)].reset_index(drop=True)
  assert not df.duplicated(['site','row','col']).any()
  df.to_parquet(out,index=False)
  print('SAVED',forest,scale,'n',len(df),'support>=' ,MIN_TEACHER_COVERAGE,flush=True)
  return df

# Always audit 50m reconstructed teacher BEFORE any other scale, using exact grid keys.
AUDIT50=[]
for forest in SITES:
 f=build_scale_site(50,forest)
 orig=D[D.site.eq(forest)][KEYS+['target_fraction']]
 joined=orig.merge(f,left_on=['site','row','col'],right_on=['site','row','col'],
                    how='inner',suffixes=('_frozen','_native'))
 assert len(joined)>=.95*len(orig),(forest,'native teacher overlap too low',len(joined),len(orig))
 diff=joined.target_fraction_native.to_numpy()-joined.target_fraction_frozen.to_numpy()
 mae=float(np.mean(abs(diff))*100)
 AUDIT50.append(dict(site=forest,n=len(joined),mae_pp=mae,
     bias_pp=float(np.mean(diff)*100),corr=float(np.corrcoef(
       joined.target_fraction_native,joined.target_fraction_frozen)[0,1])))
 assert mae<.5,(forest,'teacher inconsistent with frozen V3.1 50m source',mae)
display(pd.DataFrame(AUDIT50).round(5))
pd.DataFrame(AUDIT50).to_csv(SCALE_RUN/'08_teacher_50m_equivalence.csv',index=False)
for scale in SCALES:
 for forest in SITES:
  if scale!=50:build_scale_site(scale,forest)
print('All native area-aggregated scale tables ready:',SCALE_RUN)


## 09 — Matched annual Sentinel-2 LOFO across spatial resolutions

**Purpose:** quantify the *resolution dependence* of the SAME annual S2 model for predicting V3.1 UAV teacher fractional cover. Primary feature family = ten annual reflectances + four indices, no temporally pooled quarters, to isolate scale from seasonality. Strict forest LOFO; capped but randomly sampled test pixels are representative of *each forest at each scale*, not matched one-to-one across scales. Each site contributes equally to summary.

**Interpretation:** different grid support changes the *target variance* itself. Compare MAE, RMSE, R² and target SD together, not just R². At 10 m, pixel footprints still represent mixtures; one spectrum is not an individual-crown signature.

In [ ]:
SCALE_SEED=20261120
MAX_TRAIN_PER_FOREST=1500
MAX_TEST_PER_FOREST=15000 # deterministic cap for rapid, unbiased within-forest sample
SCALE_FEATURES=[f'S2_{j:02d}' for j in range(1,15)]
SCALE_MODELS=['annual_reflectance_indices']
SCALE_MODEL_ARGS=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                      min_samples_leaf=35,l2_regularization=2,early_stopping=False)
scale_metrics=[];all_out=[]
for scale in SCALES:
 samples={s:pd.read_parquet(SCALE_RUN/f'{s}_{scale}m.parquet') for s in SITES}
 for held in SITES:
  rng=np.random.default_rng(SCALE_SEED+scale*31+sum(map(ord,held)))
  target=samples[held]
  test_ids=np.arange(len(target))
  if len(test_ids)>MAX_TEST_PER_FOREST:
   test_ids=np.sort(rng.choice(test_ids,MAX_TEST_PER_FOREST,replace=False))
  training=[]
  for s in SITES:
   if s==held:continue
   other=samples[s]
   # Forest balanced training; select up to cap, stratified by abundance using training labels only.
   ids=np.arange(len(other))
   strata=np.digitize(other.target_fraction.to_numpy(),[.10,.20,.40])
   perbin=[ids[strata==i] for i in range(4)]
   pick=[]
   for b in perbin:
    if len(b):
     n=min(len(b),max(1,MAX_TRAIN_PER_FOREST//4))
     pick.extend(rng.choice(b,n,replace=False).tolist())
   remaining=MAX_TRAIN_PER_FOREST-len(pick)
   if remaining>0:
    rest=np.setdiff1d(ids,np.asarray(pick,int))
    if len(rest):pick.extend(rng.choice(rest,min(remaining,len(rest)),replace=False).tolist())
   training.append(other.iloc[np.asarray(sorted(pick),int)])
  fit_data=pd.concat(training,ignore_index=True)
  output=SCALE_RUN/f'09_LOFO_{scale}m_{held}.parquet'
  if output.is_file():
   pred_frame=pd.read_parquet(output)
   assert len(pred_frame)==len(test_ids)
   print('CACHE LOFO',scale,held,flush=True)
  else:
   model=HistGradientBoostingRegressor(random_state=SCALE_SEED,**SCALE_MODEL_ARGS)
   model.fit(fit_data[SCALE_FEATURES].to_numpy(np.float32),
             fit_data.target_fraction.to_numpy(np.float32))
   te=target.iloc[test_ids]
   pred=np.clip(model.predict(te[SCALE_FEATURES].to_numpy(np.float32)),0,1)
   pred_frame=te[['site','scale_m','row','col','target_fraction']].copy().reset_index(drop=True)
   pred_frame['prediction']=pred
   pred_frame.to_parquet(output,index=False)
   print('FIT LOFO',scale,held,'test',len(pred_frame),flush=True)
  ytrue=pred_frame.target_fraction.to_numpy(float)
  ypred=pred_frame.prediction.to_numpy(float)
  err=ypred-ytrue
  var=np.var(ytrue)
  scale_metrics.append(dict(scale_m=scale,site=held,n_test=len(pred_frame),
    n_train=len(fit_data),target_mean_pp=float(np.mean(ytrue)*100),
    target_sd_pp=float(np.std(ytrue)*100),
    MAE_pp=float(np.mean(abs(err))*100),RMSE_pp=float(np.sqrt(np.mean(err**2))*100),
    bias_pp=float(np.mean(err)*100),
    R2=float(1-np.mean(err**2)/var) if var>1e-12 else np.nan))
  all_out.append(pred_frame)
SCALE_PER=pd.DataFrame(scale_metrics)
SCALE_PER.to_csv(SCALE_RUN/'09_LOFO_per_forest_scale.csv',index=False)
# Compare equal-forest scores at each spatial support, incl. correct pooled variance.
equal=[]
for scale,g in SCALE_PER.groupby('scale_m'):
 pred=pd.concat([p for p in all_out if int(p.scale_m.iloc[0])==scale],ignore_index=True)
 means=pred.groupby('site').target_fraction.mean()
 overall=float(means.mean())
 variance=np.mean([np.mean((q.target_fraction.to_numpy()-overall)**2)
                       for _,q in pred.groupby('site')])
 mse=np.mean([np.mean((q.prediction.to_numpy()-q.target_fraction.to_numpy())**2)
                   for _,q in pred.groupby('site')])
 equal.append(dict(scale_m=scale,forests=len(g),
    equal_forest_R2=float(1-mse/variance) if variance>0 else np.nan,
    equal_forest_MAE_pp=float(g.MAE_pp.mean()),
    equal_forest_RMSE_pp=float(np.sqrt(np.mean(g.RMSE_pp**2))),
    equal_forest_target_SD_pp=float(np.sqrt(variance)*100),
    min_site_R2=float(g.R2.min()),MOS_R2=float(g[g.site.eq('MOS')].R2.iloc[0])))
SCALE_SUM=pd.DataFrame(equal).sort_values('scale_m')
SCALE_SUM.to_csv(SCALE_RUN/'09_LOFO_equal_forest_scale_summary.csv',index=False)
display(SCALE_SUM.round(3))
display(SCALE_PER.pivot(index='scale_m',columns='site',values='R2').round(3))
print('Scale prediction outputs:',SCALE_RUN)


## 10 — Scale curves and interpretation

Plot forest-equal R², absolute error and changes in reference target variance together. The **optimal scale** is not automatically the scale with highest R²: smoothing can inflate R² and suppress abundance extremes. Also inspect whether MOS responds differently from the other three forests.

This test holds raw annual input type and model family fixed, but changes predictor/target spatial support. It does **not** include Planet or seasonal imagery yet. Never claim performance at a scale that has not produced genuine observed-versus-predicted points.

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.8),layout='constrained')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_R2,'o-',label='Equal forest')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.MOS_R2,'s--',label='MOS only')
ax[0].set(ylabel='LOFO R²',title='Geographic transfer')
ax[0].axhline(0,ls=':',c='grey');ax[0].legend()
ax[1].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_MAE_pp,'o-')
ax[1].set(ylabel='MAE (percentage points)',title='Absolute abundance error')
ax[2].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_target_SD_pp,'o-')
ax[2].set(ylabel='Target SD (percentage points)',title='Target variability changes with scale')
for a in ax:
 a.set_xlabel('Aggregation scale (m)')
 a.set_xticks(SCALE_SUM.scale_m.to_numpy())
 a.grid(alpha=.15)
fig.suptitle('Native V3.1 UAV teacher + annual S2: honest multiresolution LOFO')
fig.savefig(SCALE_RUN/'10_multiresolution_LOFO_three_metrics.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
fig,axes=plt.subplots(2,2,figsize=(11,9),layout='constrained')
for axis,forest in zip(axes.flat,SITES):
 q=SCALE_PER[SCALE_PER.site.eq(forest)].sort_values('scale_m')
 axis.plot(q.scale_m,q.R2,'o-',label='R²')
 axis.axhline(0,ls='--',lw=.8,color='gray')
 axis.set(title=forest,xlabel='Aggregation scale (m)',ylabel='Held-out forest R²')
 axis.set_xticks(q.scale_m)
 axis.grid(alpha=.15)
fig.savefig(SCALE_RUN/'10_multiresolution_per_forest_R2.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Interpretation: choose a stable resolution based on MAE, forest transfer, and ecological map utility;')
print('avoid choosing scale by one inflated pooled R².')
